# HealthConnect Clinic — Week 4: Machine Learning Problem Definition & System Design

**Intern:** Rudolf  
**Role:** Junior Machine Learning Engineer / Data Scientist  
**Track:** Machine Learning / Data Science / ML Engineering  
**Project:** HealthConnect Clinic Experience Lab  
**Date:** August 2026  

---

## Executive Summary
This notebook presents the complete **Week 4 HealthConnect Experience Lab** deliverables for the Machine Learning track. HealthConnect Clinic faces significant operational inefficiencies caused by patient appointment no-shows and unoptimized support workflows.

In this notebook, we:
1. Conduct an in-depth **Data Quality & Suitability Assessment** on 5,000 historical appointment records.
2. Define the **Machine Learning Problem**, formulate the binary target variable (), detail cancellation handling logic, and specify a 6-category input feature taxonomy while establishing data leakage safeguards.
3. Generate diagnostic visual insights into no-show risk drivers (lead time, historical attendance, reminder channels).
4. Outline an end-to-end **Production ML System Architecture**, specifying API input/output schemas, operational workflows, and MLOps/reproducibility governance.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual aesthetic
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('HealthConnect_Appointment_Data.csv')
print(f'Dataset successfully loaded. Total rows: {df.shape[0]}, Total columns: {df.shape[1]}')
df.head(5)


Dataset successfully loaded. Total rows: 5000, Total columns: 18


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


## 1. Data Quality & Suitability Assessment

### 1.1 Structural Overview & Data Types
We analyze the schema, missingness, and uniqueness of key identifiers across the dataset.


In [2]:
# Detailed data info
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Null Count': df.isnull().sum(),
    'Null Percentage (%)': (df.isnull().sum() / len(df)) * 100,
    'Unique Values': df.nunique()
})
info_df


,Data Type,Null Count,Null Percentage (%),Unique Values
appointment_id,str,0,0.00,5000
patient_id,str,0,0.00,1696
gender,str,0,0.00,3
age,int64,0,0.00,63
age_group,str,0,0.00,6
appointment_type,str,0,0.00,4
booking_date,str,0,0.00,593
appointment_date,str,0,0.00,545
appointment_day,str,0,0.00,7
appointment_time,str,0,0.00,3


### 1.2 Target Variable Breakdown & Cancellation Logic
The target column  contains three distinct states:
- **Attended**: Patient attended their scheduled appointment ( = 2,314, 46.28\%$).
- **No-Show**: Patient failed to attend without prior notice ( = 2,423, 48.46\%$).
- **Cancelled**: Patient cancelled the appointment in advance ( = 263, 5.26\%$).

**Decision on Cancellations:**
For binary predictive modeling ( \in \{0, 1\}$), **Cancelled** appointments are filtered out from the primary no-show model dataset. Cancellations represent explicit notification where clinic resources can be re-allocated prior to the appointment, whereas No-Shows represent unannounced non-attendance resulting in idle clinical staff and lost slots.


In [3]:
# Outcome distribution analysis
outcome_counts = df['appointment_outcome'].value_counts()
outcome_pct = df['appointment_outcome'].value_counts(normalize=True) * 100
pd.DataFrame({'Count': outcome_counts, 'Percentage (%)': outcome_pct.round(2)})


,Count,Percentage (%)
appointment_outcome,,
No-Show,2423,48.46
Attended,2314,46.28
Cancelled,263,5.26


### 1.3 Missing Value Analysis & Mechanisms
1. ****: 1,366 missing values (7.32\%$). This is Not Missing At Random (NMAR) because missing values correspond exactly to appointments where .
2. ****: 90 missing values (.80\%$). Missing Completely at Random (MCAR), suitable for median imputation within patient zip code / age groups.
3. ****: 60 missing values (.20\%$). Missing At Random (MAR). *Critical Note:* Waiting time occurs *during/after* appointment arrival and must be excluded as an input feature to prevent data leakage.


In [4]:
# Verify missingness relationship between reminder_sent and reminder_channel
print('Cross-tabulation of reminder_sent vs reminder_channel missingness:')
pd.crosstab(df['reminder_sent'], df['reminder_channel'].isnull(), rownames=['Reminder Sent'], colnames=['Channel Missing'])


Cross-tabulation of reminder_sent vs reminder_channel missingness:


Channel Missing,False,True
Reminder Sent,,
No,0,1366
Yes,3634,0


## 2. Longitudinal Patient Structure & Multi-Appointment Analysis

The 5,000 appointment records belong to **1,696 unique patients** (), averaging **2.95 appointments per patient**.

**ML System Implication:**
Because patients appear multiple times, standard random train/test splits would cause **data leakage across time**. Production models must employ **GroupKFold** cross-validation grouped by  or a **Temporal Train-Test Split** based on  to reflect real-world deployment.


In [5]:
patient_appt_counts = df['patient_id'].value_counts()
print(f'Total Unique Patients: {df["patient_id"].nunique()}')
print(f'Min Appointments per Patient: {patient_appt_counts.min()}')
print(f'Max Appointments per Patient: {patient_appt_counts.max()}')
print(f'Average Appointments per Patient: {patient_appt_counts.mean():.2f}')


Total Unique Patients: 1696
Min Appointments per Patient: 1
Max Appointments per Patient: 9
Average Appointments per Patient: 2.95


## 3. Machine Learning Problem Definition & Feature Taxonomy

### 3.1 Problem Formulation
We formulate the problem as a **Supervised Binary Classification** task:
$33643P(Y = 1 \mid X)33643$
where  = 1$ denotes a **No-Show**,  = 0$ denotes **Attended**, and $ is the vector of pre-appointment features available at the scoring execution window ( - 48$ hours prior to scheduled appointment).

### 3.2 Candidate Input Feature Categories
1. **Patient Demographics:** , , .
2. **Appointment Characteristics:** , , .
3. **Temporal & Lead Time Features:** , , day of week, day of month.
4. **Historical Attendance Behavior:** , , engineered .
5. **Engagement & Communication:** ,  (encoded as categorical with  category).
6. **Logistics & Accessibility:**  (imputed).


## 4. Key Exploratory Diagnostic Insights

In [6]:
# Filter binary modeling subset
df_model = df[df['appointment_outcome'].isin(['Attended', 'No-Show'])].copy()
df_model['is_no_show'] = (df_model['appointment_outcome'] == 'No-Show').astype(int)

# Lead time impact
df_model['lead_group'] = pd.cut(df_model['booking_lead_days'], 
                                bins=[-1, 3, 7, 14, 30, 60, 365], 
                                labels=['Same/0-3d', '4-7d', '8-14d', '15-30d', '31-60d', '60d+'])

lead_summary = df_model.groupby('lead_group', observed=False)['is_no_show'].agg(['count', 'mean']).reset_index()
lead_summary['no_show_rate_pct'] = lead_summary['mean'] * 100
lead_summary


,lead_group,count,mean,no_show_rate_pct
0,Same/0-3d,297,0.265993,26.599327
1,4-7d,307,0.322476,32.247557
2,8-14d,574,0.351916,35.191638
3,15-30d,1265,0.455336,45.533597
4,31-60d,2294,0.639494,63.949433
5,60d+,0,NaN,NaN


In [7]:
# Historical No-Shows impact
prev_summary = df_model.groupby('previous_no_shows', observed=False)['is_no_show'].agg(['count', 'mean']).reset_index()
prev_summary['no_show_rate_pct'] = prev_summary['mean'] * 100
prev_summary.head(6)


,previous_no_shows,count,mean,no_show_rate_pct
0,0,2745,0.463024,46.302368
1,1,1482,0.558704,55.870445
2,2,419,0.620525,62.052506
3,3,76,0.697368,69.736842
4,4,12,0.666667,66.666667
5,5,3,1.000000,100.000000


## 5. End-to-End ML System Design & Production Architecture

### 5.1 System Workflow & Pipeline Architecture
The HealthConnect ML solution follows a modular 5-stage architecture:
1. **Data Ingestion & Event Trigger:** Batch ETL extracts upcoming appointments scheduled for  + 2$ days.
2. **Data Validation & Feature Store:** Schema enforcement, missing value imputation, historical feature retrieval.
3. **ML Inference Service:** LightGBM / XGBoost classifier calculates probability (	ext{No-Show})$.
4. **Actionable Decision Engine:**
   - **High Risk ( > 0.70$):** Trigger automated WhatsApp / SMS confirmation request + reserve standby patient buffer slot.
   - **Moderate Risk (bash.40 \le P \le 0.70$):** Automated SMS reminder with interactive reply choice.
   - **Low Risk ( < 0.40$):** Standard email notification.
5. **MLOps & Drift Monitoring:** Real-time logging of prediction scores vs observed outcomes in MLflow & Prometheus.


## 6. Assumptions, Risks & Week 5 Roadmap

### 6.1 Key Risk Matrix
- **Data Leakage Risk:**  must be strictly excluded from real-time prediction features.
- **Fairness & Bias:** Ensure model performance (Recall & False Positive Rate) is balanced across demographic subgroups (, ).
- **Cold Start Problem:** New patients without  history require demographic and lead-time baseline priors.

### 6.2 Proposed Week 5 Focus
- Implement custom transformers for imputation and feature engineering.
- Train baseline classification models (Logistic Regression, Random Forest, XGBoost, LightGBM).
- Perform hyperparameter optimization and probability calibration.
